In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing import image
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
dataset_path = "/content/drive/MyDrive/face_analysis/Datasets"

In [ ]:
IMG_SIZE = (224,224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

In [ ]:
class_names = train_ds.class_names

print(class_names)

In [ ]:
# Display Sample Images

plt.figure(figsize=(12, 12))

for images, labels in train_dataset.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")

plt.show()

In [ ]:
# Optimize Dataset Performance

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

In [ ]:
# Load Pre-trained MobileNetV2

base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze the base model
base_model.trainable = False

# Build the complete model
model = Sequential([
    layers.Rescaling(1./255),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(len(class_names), activation="softmax")
])

# Display model summary
model.summary()

In [ ]:
# Compile the Model

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
# Early Stopping Callback This prevents overfitting by stopping training if the validation accuracy stops improving

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

In [ ]:
# Train the Model

history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=10,
    callbacks=[early_stopping]
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,5))

# Accuracy
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training vs Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Loss
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
print("Evaluating model...")

loss, accuracy = model.evaluate(validation_dataset)

print(f"\nValidation Loss: {loss:.4f}")
print(f"Validation Accuracy: {accuracy:.4f}")

In [ ]:
# Save the trained model

model.save("/content/drive/MyDrive/face_analysis/facial_skin_model.keras")

print("Model saved successfully!")

In [ ]:
 from google.colab import files

uploaded = files.upload()

In [ ]:
from tensorflow.keras.preprocessing import image
import numpy as np
import matplotlib.pyplot as plt

# Get uploaded file name
img_path = list(uploaded.keys())[0]

# Load and preprocess image
img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)


prediction = model.predict(img_array)

predicted_class = class_names[np.argmax(prediction)]
confidence = np.max(prediction) * 100

plt.imshow(img)
plt.axis("off")
plt.title(f"Prediction: {predicted_class}\nConfidence: {confidence:.2f}%")
plt.show()

print(f"Predicted Skin Condition: {predicted_class}")
print(f"Confidence: {confidence:.2f}%")

In [ ]:
# Generate predictions on validation dataset

y_true = []
y_pred = []

for images, labels in validation_dataset:
    predictions = model.predict(images, verbose=0)
    predicted_labels = np.argmax(predictions, axis=1)

    y_true.extend(labels.numpy())
    y_pred.extend(predicted_labels)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

In [ ]:
print("Classification Report\n")

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")

plt.show()

In [ ]:
from google.colab import files

print("📤 Upload BEFORE image")
before_uploaded = files.upload()

print("\n📤 Upload AFTER image")
after_uploaded = files.upload()

before_path = list(before_uploaded.keys())[0]
after_path = list(after_uploaded.keys())[0]

In [ ]:
from tensorflow.keras.preprocessing import image
import numpy as np

def predict_image(img_path):

    img = image.load_img(img_path, target_size=(224,224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array, verbose=0)

    predicted_index = np.argmax(prediction)

    predicted_class = class_names[predicted_index]

    confidence = prediction[0][predicted_index] * 100

    return predicted_class, confidence, prediction[0]

In [ ]:
before_class, before_confidence, before_scores = predict_image(before_path)

after_class, after_confidence, after_scores = predict_image(after_path)

In [ ]:
import matplotlib.pyplot as plt

before_img = image.load_img(before_path)
after_img = image.load_img(after_path)

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.imshow(before_img)
plt.title("Before Image")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(after_img)
plt.title("After Image")
plt.axis("off")

plt.show()

In [ ]:
print("="*60)
print("BEFORE IMAGE - PROBABILITY OF ALL CLASSES")
print("="*60)

for cls, prob in zip(class_names, before_scores):
    print(f"{cls:<15} : {prob*100:.2f}%")

print("\nPredicted Class :", before_class)

In [ ]:
print("="*60)
print("AFTER IMAGE - PROBABILITY OF ALL CLASSES")
print("="*60)

for cls, prob in zip(class_names, after_scores):
    print(f"{cls:<15} : {prob*100:.2f}%")

print("\nPredicted Class :", after_class)

In [ ]:
import pandas as pd

comparison_data = []

for cls, before_prob, after_prob in zip(class_names, before_scores, after_scores):

    before_percent = before_prob * 100
    after_percent = after_prob * 100

    difference = after_percent - before_percent

    if difference > 0:
        change = f"↑ {difference:.2f}%"
    elif difference < 0:
        change = f"↓ {abs(difference):.2f}%"
    else:
        change = "No Change"

    comparison_data.append([
        cls,
        f"{before_percent:.2f}%",
        f"{after_percent:.2f}%",
        change
    ])

comparison_df = pd.DataFrame(
    comparison_data,
    columns=["Skin Condition", "Before", "After", "Change"]
)

print("="*70)
print("BEFORE vs AFTER COMPARISON")
print("="*70)

display(comparison_df)

In [ ]:
print("="*70)
print("MODEL INTERPRETATION")
print("="*70)

print(f"Before Image Prediction : {before_class} ({before_confidence:.2f}%)")
print(f"After Image Prediction  : {after_class} ({after_confidence:.2f}%)")
print()

if before_class != after_class:
    print(f"✓ The model detected a change from '{before_class}' to '{after_class}'.")
else:
    print(f"✓ The model predicted the same condition ('{before_class}') in both images.")

print()

highest_before = before_scores.max()*100
highest_after = after_scores.max()*100

print(f"Highest confidence (Before): {highest_before:.2f}%")
print(f"Highest confidence (After) : {highest_after:.2f}%")